# 02 · 카드 생성 파이프라인

**실행 순서:** 셀0(설정) → 셀1(데이터 로드) → 셀2(파이프라인 초기화) → 셀3(테스트) → 셀4(결과 확인) → 셀5(전체 실행)

> preprocessed 폴더의 JSON/JSONL 파일을 직접 읽어 카드를 생성합니다. DB 없이도 동작합니다.

## 셀 0. 설정

In [1]:
import sys, json, logging
from pathlib import Path

# 경로 설정
PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "pipeline":
    PIPELINE_DIR = PIPELINE_DIR / "pipeline"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "embedding_hf"))

from openai import OpenAI
from sqlalchemy import create_engine
from config import OPENAI_API_KEY, DB_URL
from db.rdb import get_engine, init_tables, load_cards, load_card_tabs
from pipelines.news_card_generator import NewsCardGenerator
from pipelines.policy_card.graph import PolicyCardGenerator

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# ── 클라이언트 초기화 ──────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)
engine = get_engine(DB_URL)
init_tables(engine)

# ── VectorDB: 카드 생성 중엔 None (SQLite 저장만, upsert 스킵)
# ── app.py에서 Qdrant를 @st.cache_resource로 한 번만 열어서 RAG 검색에 사용
vdb = None

# ── 경로 설정 ──────────────────────────────────────────────────────────────
BASE_PRE     = PIPELINE_DIR.parent / "data" / "preprocessed"
ARTICLE_JSON = BASE_PRE / "article" / "news_20260526_pretty.json"
POLICY_DIR   = BASE_PRE / "policy" / "policies_json"
BILL_DIR     = BASE_PRE / "bill" / "bills_markdown"

print("✅ 설정 완료")
print(f"  ARTICLE_JSON : {ARTICLE_JSON.exists()}")
print(f"  POLICY_DIR   : {POLICY_DIR.exists()}")
print(f"  BILL_DIR     : {BILL_DIR.exists()}")


DB 테이블 초기화 완료
✅ 설정 완료
  ARTICLE_JSON : True
  POLICY_DIR   : True
  BILL_DIR     : True


## 셀 1. preprocessed 데이터 로드

In [2]:
# ── 기사 로드 (JSON 배열) ─────────────────────────────────────────────────
import json

with open(ARTICLE_JSON, encoding="utf-8") as f:
    ALL_ARTICLES = json.load(f)

print(f"기사 로드: {len(ALL_ARTICLES)}건")
print(f"  키: {list(ALL_ARTICLES[0].keys())}")

# ── 정책 로드 (JSON) ──────────────────────────────────────────────────────
ALL_POLICIES = []
for p in sorted(POLICY_DIR.glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    ALL_POLICIES.append({
        "id":      p.stem,
        "name":    d.get("서비스명", ""),
        "content": d.get("지원내용", ""),
        "target":  d.get("지원대상", ""),
        "method":  d.get("신청방법", ""),
        "period":  d.get("신청기한", ""),
        "contact": d.get("전화문의", ""),
        "org":     d.get("소관기관명", ""),
        "url":     "",
        "_raw":    d,
    })

print(f"\n정책 로드: {len(ALL_POLICIES)}건")
for p in ALL_POLICIES:
    print(f"  - {p['name']}")

# ── 법안 로드 (마크다운 .md) ──────────────────────────────────────────────
ALL_BILLS = []
for b in sorted(BILL_DIR.glob("*.md")):
    md_content = b.read_text(encoding="utf-8")
    # 첫 줄에서 법안명 추출 (# 제목 형식)
    lines = [l.strip() for l in md_content.splitlines() if l.strip()]
    name  = lines[0].lstrip("#").strip() if lines else b.stem
    ALL_BILLS.append({
        "id":      b.stem,
        "name":    name,
        "content": md_content[:4000],
        "url":     "",
    })

print(f"\n법안 로드: {len(ALL_BILLS)}건")
if ALL_BILLS:
    print(f"  샘플: {ALL_BILLS[0]['name'][:50]}")


기사 로드: 153건
  키: ['title', 'content', 'publisher', 'published_at', 'url', 'source', 'keyword_matched', 'file_path']

정책 로드: 5건
  - 국민취업지원제도
  - 취약계층을 위한 공공일자리 제공
  - 전남 청년 근속장려금 지원사업
  - 세종시 청년 구직활동비 지원
  - 미래 청년 일자리

법안 로드: 122건
  샘플: 법률명


## 셀 2. 파이프라인 초기화

In [3]:
# ── 생성기 초기화 (vdb=None → SQLite 저장만, Qdrant upsert 스킵) ──────────
news_gen   = NewsCardGenerator(engine, vdb, client)
policy_gen = PolicyCardGenerator(engine, vdb, client)

print("✅ 생성기 초기화 완료")
print("  카드 생성 후 SQLite(politalk_dev.db)에 저장됩니다.")
print("  Qdrant cards 컬렉션 upsert는 app.py 실행 시 자동 처리됩니다.")


✅ 생성기 초기화 완료
  카드 생성 후 SQLite(politalk_dev.db)에 저장됩니다.
  Qdrant cards 컬렉션 upsert는 app.py 실행 시 자동 처리됩니다.


## 셀 3. 단건 테스트

개별 생성기로 카드 1장씩 빠르게 확인합니다.

In [ ]:
# ════════════════════════════════════════════════════════
# 3-1. 정책 카드 단건 테스트 (save=True → DB 저장)
# ════════════════════════════════════════════════════════
POLICY_SOURCE    = ALL_POLICIES[0]   # 0~4 중 선택
RELATED_ARTICLES = ALL_ARTICLES[:5]

print(f"\n{'='*55}")
print(f"🧪 정책 카드 단건 테스트")
print(f"  정책명  : {POLICY_SOURCE['name']}")
print(f"  관련기사: {len(RELATED_ARTICLES)}건")
print(f"{'='*55}\n")

policy_result = policy_gen.run(
    source           = POLICY_SOURCE,
    related_articles = RELATED_ARTICLES,
    card_type        = "POLICY",
    save             = True,   # DB 저장
)

if policy_result:
    print(f"\n{'='*55}")
    print("✅ 정책 카드 생성 완료")
    cd      = policy_result["tabs"]
    summary = cd.get("SUMMARY", {})
    opinion = cd.get("OPINION", [])
    print(f"  card_id : {policy_result.get('card_id', '미저장')}")
    print(f"  제목    : {summary.get('title', '')}")
    print(f"  카테고리: {summary.get('category', '')}")
    print(f"  요약    : {summary.get('summary_points', [])}")
    print(f"  CORE   : {len(cd.get('CORE',''))}자")
    print(f"  OPINION: {len(opinion)}개 입장")
else:
    print("❌ 정책 카드 생성 실패")


In [ ]:
# ════════════════════════════════════════════════════════
# 3-2. 법안 카드 단건 테스트
# ════════════════════════════════════════════════════════
BILL_SOURCE = ALL_BILLS[0]   # 인덱스 조정 가능

print(f"\n{'='*55}")
print(f"🧪 법안 카드 단건 테스트")
print(f"  법안명: {BILL_SOURCE['name']}")
print(f"{'='*55}\n")

bill_result = policy_gen.run(
    source           = BILL_SOURCE,
    related_articles = ALL_ARTICLES[:3],
    card_type        = "BILL",
    save             = False,
)

if bill_result:
    print(f"\n{'='*55}")
    print("✅ 법안 카드 생성 완료")
    cd = bill_result["tabs"]
    summary = cd.get("SUMMARY", {})
    print(f"  제목    : {summary.get('title', '')}")
    print(f"  카테고리: {summary.get('category', '')}")
    print(f"  CORE   : {len(cd.get('CORE',''))}자")
    print(f"  OPINION: {len(cd.get('OPINION', []))}개 입장")
else:
    print("❌ 법안 카드 생성 실패 (편향 감지 또는 오류)")


In [ ]:
# ════════════════════════════════════════════════════════
# 3-3. 뉴스 카드 단건 테스트 (save=True → DB 저장)
# ════════════════════════════════════════════════════════
NEWS_CLUSTER = ALL_ARTICLES[:5]   # 클러스터처럼 기사 5건 묶어서

print(f"\n{'='*55}")
print(f"🧪 뉴스 카드 단건 테스트")
print(f"  기사 수: {len(NEWS_CLUSTER)}건")
for i, a in enumerate(NEWS_CLUSTER, 1):
    print(f"  [{i}] {a.get('publisher','?')} — {a.get('title','')[:40]}")
print(f"{'='*55}\n")

news_result = news_gen.run(
    articles = NEWS_CLUSTER,
    save     = True,   # DB 저장
)

if news_result:
    print(f"\n{'='*55}")
    print("✅ 뉴스 카드 생성 완료")
    cd      = news_result["tabs"]
    summary = cd.get("SUMMARY", {})
    opinion = cd.get("OPINION", [])
    print(f"  card_id : {news_result.get('card_id', '미저장')}")
    print(f"  제목    : {summary.get('title', '')}")
    print(f"  카테고리: {summary.get('category', '')}")
    print(f"  요약    : {summary.get('summary_points', [])}")
    print(f"  CORE   : {len(cd.get('CORE',''))}자")
    print(f"  OPINION: {len(opinion)}개 언론사")
else:
    print("❌ 뉴스 카드 생성 실패")


## 셀 4. 생성 결과 상세 확인

In [ ]:
# 마지막으로 생성된 카드 전체 내용 출력
# policy_result / bill_result / news_result 중 확인할 것 선택
TARGET = news_result   # ← 여기 변경 (policy_result / bill_result / news_result)

if TARGET and TARGET.get("tabs"):
    tabs = TARGET["tabs"]
    print(f"\n카드 타입: {TARGET.get('card_type')} | 제목: {TARGET.get('title')}\n")

    for tab_name, content in tabs.items():
        print(f"{'='*55}")
        print(f"[{tab_name}]")
        if isinstance(content, str):
            print(content[:1000])
            if len(content) > 1000:
                print(f"  ... (총 {len(content)}자)")
        elif isinstance(content, (dict, list)):
            print(json.dumps(content, ensure_ascii=False, indent=2)[:1000])
        print()


In [ ]:
# DB에 저장된 카드 확인 (save=True 로 실행한 경우)
cards = load_cards(engine, status="DRAFT", limit=10)
print(f"DRAFT 카드: {len(cards)}개")

for c in cards[:5]:
    print(f"  #{c['id']} [{c['card_type']}] {c.get('created_at','')}")

if cards:
    card_id = cards[0]["id"]
    tabs = load_card_tabs(engine, card_id)
    print(f"\n카드 #{card_id} 탭 목록: {list(tabs.keys())}")
    for tab_type, content in tabs.items():
        try:
            parsed = json.loads(content)
            preview = json.dumps(parsed, ensure_ascii=False, indent=2)[:200]
        except Exception:
            preview = str(content)[:200]
        print(f"\n[{tab_type}]\n{preview}...")


## 셀 5. 배치 실행 (정책/법안 전체)

In [5]:
from tqdm import tqdm

# ── 5-1. 정책 카드 전체 생성 (모든 정책 → DB 저장) ──────────────────────
print(f"📂 발견된 정책 파일: {len(ALL_POLICIES)}개 — 전체 생성합니다")

policy_cards = []
print("\n🚀 정책 카드 생성 중:")
for source in tqdm(ALL_POLICIES, desc="정책 카드"):
    related = ALL_ARTICLES[:5]
    result  = policy_gen.run(source, related, card_type="POLICY", save=True)
    if result:
        policy_cards.append(result)
        print(f"  ✅ card_id={result.get('card_id')} | {result['title']}")
    else:
        print(f"  ❌ {source['name']} — 생성 실패")

print(f"\n정책 카드 완료: {len(policy_cards)}/{len(ALL_POLICIES)}개")
print("저장된 card_id 목록:", [r.get('card_id') for r in policy_cards])


📂 발견된 정책 파일: 5개 — 전체 생성합니다

🚀 정책 카드 생성 중:


정책 카드:   0%|          | 0/5 [00:00<?, ?it/s]01:48:22 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:26 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:30 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:34 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:42 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:51 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:51 [INFO] 📋 [extract_facts] 정책: '국민취업지원제도' | 기사: 5건


📋 [extract_facts] 정책: '국민취업지원제도' | 기사: 5건


01:48:56 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:48:56 [INFO] 📝 [generate_summary] title='맞춤형 취업지원 서비스'


📝 [generate_summary] title='맞춤형 취업지원 서비스'


...

🗂️  [assemble] 카드 조립 완료: '미래 청년 일자리 매칭 프로그램'


01:53:25 [INFO] 💾 [save] POLICY 카드 #5 저장 완료


💾 [save] POLICY 카드 #5 저장 완료


정책 카드: 100%|██████████| 5/5 [05:09<00:00, 61.89s/it]

  ✅ card_id=5 | 미래 청년 일자리 매칭 프로그램

정책 카드 완료: 5/5개
저장된 card_id 목록: [1, 2, 3, 4, 5]


In [6]:
# ── 5-2. 법안 카드 배치 생성 ─────────────────────────────────────────────
# 법안이 많으므로 상위 N개만 생성
BILL_LIMIT = 5   # 조정 가능

print(f"📂 발견된 법안 파일 개수: {len(ALL_BILLS)}개 (상위 {BILL_LIMIT}개 생성)")

bill_cards = []
print("\n🚀 법안 카드 생성 중:")
for source in tqdm(ALL_BILLS[:BILL_LIMIT], desc="법안 카드"):
    related = ALL_ARTICLES[:5]
    result = policy_gen.run(source, related, card_type="BILL", save=True)
    if result:
        bill_cards.append(result)
        print(f"  ✅ [{result['card_type']}] {result['title']}")
    else:
        print(f"  ❌ {source['name']} — 생성 실패")

print(f"\n법안 카드 완료: {len(bill_cards)}/{min(BILL_LIMIT, len(ALL_BILLS))}개")


📂 발견된 법안 파일 개수: 122개 (상위 5개 생성)

🚀 법안 카드 생성 중:


법안 카드:   0%|          | 0/5 [00:00<?, ?it/s]01:53:32 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:35 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:38 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:41 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:45 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:50 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:50 [INFO] 📋 [extract_facts] 정책: '일하는 사람 기본법안' | 기사: 5건


📋 [extract_facts] 정책: '일하는 사람 기본법안' | 기사: 5건


01:53:52 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:53:52 [INFO] 📝 [generate_summary] title='모든 일하는 사람을 위한 보호법'


📝 [generate_summary] title='모든 일하는 사람을 위한 보호법'


...

🗂️  [assemble] 카드 조립 완료: '일·가정 양립을 위한 법 개정'


01:57:54 [INFO] 💾 [save] BILL 카드 #10 저장 완료


💾 [save] BILL 카드 #10 저장 완료


법안 카드: 100%|██████████| 5/5 [04:28<00:00, 53.76s/it]

  ✅ [BILL] 일·가정 양립을 위한 법 개정

법안 카드 완료: 5/5개


In [7]:
# ── 5-3. 뉴스 카드 배치 생성 ─────────────────────────────────────────────
# ALL_ARTICLES를 N개씩 묶어 클러스터로 취급
from tqdm import tqdm

CLUSTER_SIZE  = 5    # 클러스터 당 기사 수 (조정 가능)
NEWS_LIMIT    = 3    # 생성할 뉴스 카드 수 (조정 가능)

# 간단한 클러스터링: 순서대로 N개씩 묶기
clusters = [
    ALL_ARTICLES[i:i+CLUSTER_SIZE]
    for i in range(0, len(ALL_ARTICLES), CLUSTER_SIZE)
    if len(ALL_ARTICLES[i:i+CLUSTER_SIZE]) >= 2   # 최소 2건 이상
]

print(f"📂 총 클러스터 수: {len(clusters)}개 (상위 {NEWS_LIMIT}개 생성)")

news_cards = []
print("\n🚀 뉴스 카드 생성 중:")
for cluster in tqdm(clusters[:NEWS_LIMIT], desc="뉴스 카드"):
    result = news_gen.run(cluster, save=True)
    if result:
        news_cards.append(result)
        print(f"  ✅ [{result['card_type']}] {result['title']} (card_id={result.get('card_id')})")
    else:
        titles = [a.get('title','?')[:20] for a in cluster[:2]]
        print(f"  ❌ 생성 실패: {titles}...")

print(f"\n뉴스 카드 완료: {len(news_cards)}/{min(NEWS_LIMIT, len(clusters))}개")


📂 총 클러스터 수: 31개 (상위 3개 생성)

🚀 뉴스 카드 생성 중:


뉴스 카드:   0%|          | 0/3 [00:00<?, ?it/s]01:57:58 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:06 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:12 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:17 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:22 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:22 [INFO]   [extract_facts] 5개 기사 사실 추출
01:58:24 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:24 [INFO]   [generate_summary] title='청년 지원 확대 소식'
01:58:28 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:28 [INFO]   [generate_core] 1021자
01:58:40 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:58:40 [INFO]   [check_bias_core] 통과
01:58:

  ✅ [NEWS] 청년 지원 확대 소식 (card_id=11)


01:58:59 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:06 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:10 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:15 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:25 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:25 [INFO]   [extract_facts] 5개 기사 사실 추출
01:59:28 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:28 [INFO]   [generate_summary] title='장애학생 위한 캠퍼스 개선'
01:59:36 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:36 [INFO]   [generate_core] 1157자
01:59:56 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:59:56 [INFO]   [check_bias_core] 통과
02:00:01 [INFO] HTTP Request: POST https://api.

  ✅ [NEWS] 장애학생 위한 캠퍼스 개선 (card_id=12)


02:00:21 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:28 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:34 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:40 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:48 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:48 [INFO]   [extract_facts] 5개 기사 사실 추출
02:00:51 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:51 [INFO]   [generate_summary] title='청년 취업을 위한 다양한 프로그램 소식'
02:00:59 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:00:59 [INFO]   [generate_core] 961자
02:01:09 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
02:01:09 [INFO]   [check_bias_core] 통과
02:01:14 [INFO] HTTP Request: POST https:

  ✅ [NEWS] 청년 취업을 위한 다양한 프로그램 소식 (card_id=13)

뉴스 카드 완료: 3/3개


In [8]:
import sqlite3, json
from pathlib import Path

db_path = PIPELINE_DIR / "politalk_dev.db"
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# 카드 목록
cur.execute("SELECT id, card_type, status, created_at FROM cards ORDER BY id")
cards = cur.fetchall()
print(f"✅ 총 카드: {len(cards)}개\n")

for card in cards:
    card_id = card["id"]
    print(f"#{card_id} [{card['card_type']}] {card['status']} | {card['created_at']}")
    
    # 탭 목록
    cur.execute("SELECT tab_type, content FROM card_tabs WHERE card_id=?", (card_id,))
    tabs = cur.fetchall()
    for tab in tabs:
        content = tab["content"]
        try:
            parsed = json.loads(content)
            title = parsed.get("title", "") if isinstance(parsed, dict) else ""
            preview = f"title={title}" if title else str(content)[:60]
        except:
            preview = str(content)[:60]
        print(f"  [{tab['tab_type']}] {preview}")
    print()

conn.close()

✅ 총 카드: 13개

#1 [POLICY] DRAFT | 2026-05-27 16:49:28
  [SUMMARY] title=맞춤형 취업지원 서비스
  [CORE] 고용 문제는 현대 사회에서 꾸준히 논의되어 온 주제입니다. 특히, 취업취약계층은 전통적인 고용보험이나 실업급
  [OPINION] [{"stance": "찬성", "argument": "고용노동부는 '국민취업지원제도'를 통해 취업취약계층에
  [SOURCE] {"url": "", "name": "국민취업지원제도"}

#2 [POLICY] DRAFT | 2026-05-27 16:50:37
  [SUMMARY] title=청년을 위한 공공일자리 기회
  [CORE] 서울특별시 영등포구가 도입한 취약계층을 위한 공공일자리 제공 정책은 지역 내 저소득층과 취약계층을 지원하기 
  [OPINION] [{"stance": "찬성", "argument": "서울특별시 영등포구의 '취약계층을 위한 공공일자리 제
  [SOURCE] {"url": "", "name": "취약계층을 위한 공공일자리 제공"}

#3 [POLICY] DRAFT | 2026-05-27 16:51:44
  [SUMMARY] title=전남 청년 근속장려금 지원
  [CORE] 전남 청년 근속장려금 지원사업은 청년의 고용 안정성과 기업의 인력 유지에 기여하기 위해 도입된 정책입니다. 
  [OPINION] [{"stance": "찬성", "argument": "전남 청년유니온은 '전남 청년 근속장려금 지원사업'이
  [SOURCE] {"url": "", "name": "전남 청년 근속장려금 지원사업"}

#4 [POLICY] DRAFT | 2026-05-27 16:52:35
  [SUMMARY] title=세종시 청년 구직 지원금
  [CORE] 최근 몇 년간 청년 실업 문제가 사회적으로 큰 이슈가 되고 있다. 청년층은 취업의 어려움을 겪고 있으며, 특
  [OPINION] [{"stance": "찬성", "argument": "세종시의 청

## 셀 6. 전체 파이프라인 실행 (뉴스 클러스터링 포함)

> 기사 클러스터링 + 뉴스/정책/법안 카드를 한 번에 생성합니다.

In [ ]:
# ── run_daily: 기사 클러스터링 + 전체 카드 생성 ───────────────────────────
# preprocessed 데이터를 DB에 먼저 적재하거나,
# pipeline 내부 load_articles() 대신 ALL_ARTICLES를 직접 넘기려면
# CardGenerationPipeline을 커스텀해야 합니다.
# 
# 지금은 DB 기반으로 실행합니다 (노트북 01 실행 후 데이터가 적재된 상태).

result_full = pipeline.run_daily(
    article_limit = 200,
    policy_limit  = len(ALL_POLICIES),
    bill_limit    = 10,
    publish       = False,
)

print(f"\n최종 결과:")
print(f"  뉴스 카드:  {len(result_full['news_cards'])}개")
print(f"  정책 카드:  {len(result_full['policy_cards'])}개")
print(f"  법안 카드:  {len(result_full['bill_cards'])}개")


In [4]:
import sys, json, logging
from pathlib import Path

# 경로 설정
PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "pipeline":
    PIPELINE_DIR = PIPELINE_DIR / "pipeline"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "embedding_hf"))

from openai import OpenAI
from sqlalchemy import create_engine
from config import OPENAI_API_KEY, DB_URL
from db.rdb import get_engine, init_tables, load_cards, load_card_tabs
from pipelines.news_card_generator import NewsCardGenerator
from pipelines.policy_card.graph import PolicyCardGenerator

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# ── 클라이언트 초기화 ──────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)
engine = get_engine(DB_URL)
init_tables(engine)

# ── VectorDB: 카드 생성 중엔 None (SQLite 저장만, upsert 스킵)
# ── app.py에서 Qdrant를 @st.cache_resource로 한 번만 열어서 RAG 검색에 사용
vdb = None

# ── 경로 설정 ──────────────────────────────────────────────────────────────
BASE_PRE     = PIPELINE_DIR.parent / "data" / "preprocessed"
ARTICLE_JSON = BASE_PRE / "article" / "news_20260526_pretty.json"
POLICY_DIR   = BASE_PRE / "policy" / "policies_json"
BILL_DIR     = BASE_PRE / "bill" / "bills_markdown"

print("✅ 설정 완료")
print(f"  ARTICLE_JSON : {ARTICLE_JSON.exists()}")
print(f"  POLICY_DIR   : {POLICY_DIR.exists()}")
print(f"  BILL_DIR     : {BILL_DIR.exists()}")


DB 테이블 초기화 완료
✅ 설정 완료
  ARTICLE_JSON : True
  POLICY_DIR   : True
  BILL_DIR     : True


In [9]:
import sqlite3
conn = sqlite3.connect(r"C:\skn24\team2_final\pipeline\politalk_dev.db")
cur = conn.cursor()
cur.execute("PRAGMA table_info(chat_sessions)")
cols = cur.fetchall()
for c in cols:
    print(c)
conn.close()

(0, 'id', 'BIGINT', 1, None, 1)
(1, 'user_id', 'BIGINT', 1, None, 0)
(2, 'active_newscard_id', 'BIGINT', 0, None, 0)
(3, 'title', 'VARCHAR(255)', 1, None, 0)
(4, 'created_at', 'DATETIME', 1, 'CURRENT_TIMESTAMP', 0)


In [11]:
import sqlite3
# 루트 DB
conn = sqlite3.connect(r"C:\skn24\team2_final\politalk_dev.db")
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("루트 DB 테이블:", cur.fetchall())
cur.execute("SELECT COUNT(*) FROM cards") if True else None
try:
    cur.execute("SELECT COUNT(*) FROM cards")
    print("루트 cards:", cur.fetchone())
except: print("루트에 cards 테이블 없음")
conn.close()

루트 DB 테이블: [('REGIONS',), ('CATEGORIES',), ('USERS',), ('INFO_CARDS',), ('RAW_DATAS',), ('USER_INTERESTS',), ('CHAT_SESSIONS',), ('BOOKMARKS',), ('DEBATES',), ('RAW_ARTICLES',), ('RAW_BILLS',), ('RAW_POLICIES',), ('CHAT_MESSAGES',), ('DEBATE_MESSAGES',), ('CHAT_MSG_CARDS',), ('articles',), ('sqlite_sequence',), ('policies',), ('bills',), ('cards',), ('card_tabs',), ('card_articles',), ('card_policies',), ('card_bills',), ('bias_logs',), ('debate_modes',), ('debate_participants',)]
루트 cards: (0,)
